# FREIA reflectivity reduction

Compute reflectivity from a sample measurement and a direct-beam measurement taken without a sample. The workflow converts events to specular Q, normalizes by an incident wavelength monitor, and divides the selected peaks to obtain R(Q).

This example uses local McStas files with WFM chopper settings. The sample and direct-beam runs must have matching slit and chopper settings.

In [ ]:
%matplotlib inline
import scipp as sc

from ess.freia import FreiaMcStasWorkflow
from ess.freia.corrections import RunNormalization
from ess.freia.types import (
    DetectorRegionOfInterest,
    IncidentMonitor,
    QDetector,
    SampleIlluminatedFraction,
    WavelengthMonitor,
)
from ess.reduce.nexus.types import GravityVector, NeXusName
from ess.reflectometry.types import (
    BeamSize,
    Filename,
    QBins,
    ReducibleData,
    ReferenceRun,
    ReflectivityOverQ,
    SampleRun,
    SampleSize,
    WavelengthBins,
)

## Select the runs

Set the sample and no-sample filenames. These paths are relative to the notebook directory. Choose an incident wavelength monitor upstream of the sample, so its intensity is independent of reflectivity. If no monitor is available, use `RunNormalization.none`; the two runs must then already share an exposure and flux scale.

These runs were simulated without gravity, so `GravityVector` is set to zero. Use the workflow default for data that includes gravity.

In [ ]:
workflow = FreiaMcStasWorkflow(run_norm=RunNormalization.monitor_histogram)
workflow[Filename[SampleRun]] = '../../../265224.h5'
workflow[Filename[ReferenceRun]] = '../../../265222.h5'
workflow[NeXusName[IncidentMonitor]] = 'GuideexitLambda'
workflow[GravityVector] = sc.vector([0.0, 0.0, 0.0], unit='m/s^2')

workflow[WavelengthBins] = sc.linspace('wavelength', 2.0, 10.0, 81, unit='angstrom')
workflow[QBins] = sc.geomspace('Q', 0.07, 0.4, 21, unit='1/angstrom')

## Inspect the reflected and direct peaks

The angle θ is measured above the sample surface and is corrected for gravity when enabled. Its sign distinguishes the two sides of the sample plane. The specular assumption gives $Q = 4\pi\sin(\theta)/\lambda$.

Inspect both distributions before selecting corresponding peaks.

In [ ]:
detectors = workflow.compute((QDetector[SampleRun], QDetector[ReferenceRun]))
angle_bins = sc.linspace('theta', -4.0, 4.0, 161, unit='deg').to(unit='rad')
profiles = {}
for label, run in [('Sample', SampleRun), ('Direct beam', ReferenceRun)]:
    detector = detectors[QDetector[run]]
    profile = detector.hist(theta=angle_bins, dim=detector.dims)
    profile.coords['theta'] = profile.coords['theta'].to(unit='deg')
    profiles[label] = profile
sc.plot(profiles, norm='log', title='Reflected and direct beams', vmin=1e4, vmax=1e8)

## Select matching regions of interest

Select the reflected peak and its corresponding direct peak separately. The example uses the beam near 3.5°. Additional bounds can be set on pixel coordinates such as `height` or `pixel_id`. Choose wavelength and Q bins to match the available statistics.

In [ ]:
workflow[DetectorRegionOfInterest[SampleRun]] = {
    'theta': (sc.scalar(3.2, unit='deg'), sc.scalar(3.9, unit='deg')),
}
workflow[DetectorRegionOfInterest[ReferenceRun]] = {
    'theta': (sc.scalar(-3.9, unit='deg'), sc.scalar(-3.2, unit='deg')),
}

## Inspect the wavelength normalization

The incident monitor corrects wavelength-dependent flux differences between runs. Its wavelength range must cover the detector data.

In [ ]:
monitors = workflow.compute(
    (WavelengthMonitor[SampleRun], WavelengthMonitor[ReferenceRun])
)
sc.plot(
    {
        'Sample monitor': monitors[WavelengthMonitor[SampleRun]],
        'Direct-beam monitor': monitors[WavelengthMonitor[ReferenceRun]],
    },
    title='Incident wavelength spectra',
)

In [ ]:
selected = workflow.compute((ReducibleData[SampleRun], ReducibleData[ReferenceRun]))
spectra = {}
for label, run in [('Sample', SampleRun), ('Direct beam', ReferenceRun)]:
    detector = selected[ReducibleData[run]]
    spectra[label] = detector.hist(
        wavelength=workflow.compute(WavelengthBins), dim=detector.dims
    )
sc.plot(spectra, title='Selected peaks after monitor normalization')

## Configure footprint correction

The Gaussian footprint model uses the sample length along the beam and the beam FWHM at the sample. It corrects only the reflected run. Different incident beams may require different widths.

A beam width has not yet been established for these data, so this example leaves the footprint correction off. Replace `beam_size = None` with a measured width, for example `sc.scalar(width_in_mm, unit='mm')`, to enable it.

In [ ]:
sample_size = sc.scalar(80.0, unit='mm')
beam_size = None

reduction = workflow.copy()
if beam_size is None:
    reduction[SampleIlluminatedFraction] = sc.scalar(1.0)
else:
    reduction[SampleSize[SampleRun]] = sample_size
    reduction[BeamSize[SampleRun]] = beam_size

## Compute reflectivity

The reference uses the negative of the direct beam’s θ to compute its corresponding reflected Q. The workflow integrates the two selected peaks into matching Q bins and divides their intensities. Both counting uncertainties propagate. Bins with no usable direct-beam intensity are masked.

In [ ]:
reflectivity = reduction.compute(ReflectivityOverQ)
covered = (~reflectivity.masks['direct_beam']).sum().value
print(f'{covered} of {reflectivity.sizes["Q"]} Q bins have direct-beam coverage.')
title = (
    'Reflectivity'
    if beam_size is not None
    else 'Reflectivity (footprint correction omitted)'
)
reflectivity.plot(norm='log', title=title)

Footprint correction is disabled in this example. Q resolution, background subtraction, finite-sample corrections, and complete ORSO export are not yet included.